## Text Generation using Recurrent Long Short Term Memory Network


LSTMs are a type of neural network that are well-suited for tasks involving sequential data such as text generation. They are particularly useful because they can remember long-term dependencies in the data which is crucial when dealing with text that often has context that spans over multiple words or sentences.

## Implementation in Python

Text generation is a part of NLP where we train our model on dataset that involves vast amount of textual data and our LSTM model will use it to train model. Here is the step by step implementation of text generation:

### Step 1: Importing Libraries

We will import the following libraries:

* TensorFlow: For building and training the deep learning model.
* NumPy: For numerical operations on arrays.
* Pandas: For loading and processing the CSV dataset.
* random, sys: Used in text generation and output handling.

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import sys
import tensorflow as tf


### 2. Loading the Dataset

The Dataset contains vast amount of textual data for training.

* `pd.read_csv()`: Reads the CSV file into a DataFrame.
* `df['text'].dropna()`: Drops rows with missing text entries.
* `" ".join()`: Concatenates all text rows into a single string for training.
* `.lower()`: Converts text to lowercase for consistency.

In [21]:
dataset = "https://raw.githubusercontent.com/itsluckysharma01/Datasets/refs/heads/main/train.csv"
## load Data through github link

# df = pd.read_csv(dataset)
df = pd.read_csv("train.csv")

In [22]:
df.head(5)

,title,text,subject,date
0,Greens say no support for Macron's EZ budget i...,BERLIN (Reuters) - None of the German parties ...,worldnews,"October 25, 2017"
1,Trump faces uphill battle to overcome court's ...,(Reuters) - U.S. President Donald Trump faces ...,politicsNews,"February 6, 2017"
2,Ukraine president denies hampering anti-corrup...,VILNIUS/KIEV (Reuters) - Ukrainian President P...,worldnews,"December 8, 2017"
3,U.S. defense chief: White House shakeup will n...,BRUSSELS (Reuters) - U.S. Defense Secretary Ji...,politicsNews,"February 14, 2017"
4,Irish government set to fall weeks before Brex...,DUBLIN (Reuters) - Ireland s minority governme...,worldnews,"November 24, 2017"


In [23]:
text = " ".join(df['text'].dropna().astype(str)).lower()

print(f"Total characters in the dataset: {len(text)}")

Total characters in the dataset: 35695884


### 3. Creating Vocabulary and Character Mappings

We will create vocabulary of unique characters and implement character to index mapping and vise-versa.

* **sorted(set(text)):** Extracts unique characters and sorts them to form the vocabulary.
* **char2idx:** Maps each character to a unique integer index.
* **idx2char:** Maps integers back to characters and is used during text generation.
* **text_as_int:** Converts the entire text into a sequence of integer indices.

In [24]:
vocab = sorted(set(text))
print(f"Vocabulary size: {len(vocab)}")

char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = np.array(vocab)
text_as_int = np.array([char2idx[c] for c in text])

Vocabulary size: 104


### 4. Pre-processing the Data 

We will ceate dataset from integer encoded text and split sequences into input and target. Then we will shuffle and divide the dataset into batches.

In [26]:
sequence_length = 100

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(sequence_length + 1, drop_remainder=True)

def split_input_target(chunk):
    return chunk[:-1], chunk[1:]
dataset = sequences.map(split_input_target)

batch_size = 64
buffer_size = 10000
dataset = dataset.shuffle(buffer_size).batch(batch_size, drop_remainder=True)

### 5. Building the LSTM Model

We will build a LSTM model with the layers and compile the model. We will be using RMSprop optimizer in this model.

* **Embedding layer:** Converts integer indices into dense vectors of length embedding_dim.
* **LSTM layer:** Processes sequences capturing temporal dependencies with rnn_units memory cells. return_sequences=True outputs sequence at each timestep.
* **Dense layer:** Produces output logits for all characters in the vocabulary to predict the next character.

In [27]:
vocab_size = len(vocab)
embedding_dim = 64
rnn_units = 128

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_shape=[None]),
    tf.keras.layers.LSTM(rnn_units, return_sequences=True, recurrent_initializer='glorot_uniform'),
    tf.keras.layers.Dense(vocab_size)
])

def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)
model.summary()

c:\Users\PANDIT JI\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 64)       │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 128)      │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 104)      │        13,416 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,888 (464.41 KB)

 Trainable params: 118,888 (464.41 KB)

 Non-trainable params: 0 (0.00 B)

### 6. Training the LSTM model

In [ ]:
EPOCHS = 20
history = model.fit(dataset, epochs=EPOCHS)

Epoch 1/20
5522/5522 ━━━━━━━━━━━━━━━━━━━━ 617s 111ms/step - loss: 2.1013
Epoch 2/20
4823/5522 ━━━━━━━━━━━━━━━━━━━━ 1:08 99ms/step - loss: 1.4945

### 7. Generating new random text
We will try to generate some texts using our model.

* **start_string:** Initial seed text to start generation.
* **temperature:** Controls randomness; lower values make output more predictable, higher values more creative.
* **model.reset_states():** Clears LSTM states before generation.
* **tf.random.categorical():** Samples the next character probabilistically from the model’s predictions.
* **Returns:** The seed text plus generated characters.

In [ ]:
def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    input_eval = [char2idx.get(s, 0) for s in start_string.lower()]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

    model.layers[1].reset_states()
    for _ in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0) / temperature
        predicted_id = tf.random.categorical(
            predictions, num_samples=1)[-1, 0].numpy()
        
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])
    
    return start_string + ''.join(text_generated)

start_string = input("Enter a seed text to start generation: ")
print(generate_text(model, start_string=start_string, num_generate=200, temperature=0.8))